# B-Score MoEngage Event Features
**Version**: v20260428 | **Model**: B-Score | **Date**: 2026-08-13

**Pipeline:**
1. Build BigQuery mapping and events source tables (3 BQ queries)
2. Load events into Pandas (single BQ read)
3. Preprocess: UTC → Manila tz-naive, observation_datetime setup
4. Compute ~300 features per loan via 11 feature modules (**parallel**)
5. Deduplicate → **1 row per `customer_id × observation_date`**
6. Write to BigQuery (`nitin_event_features`) + local CSV backup

**Performance vs main.py:** Pre-grouped O(1) loan lookup + `ThreadPoolExecutor` parallel processing.

## 1  Imports

In [1]:
import os
import sys
import time
import warnings
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from google.cloud import bigquery

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2  Configuration
> **Update `OBDATE` for each scoring run.** All table names are derived automatically.

In [2]:

# B Score model configuration
# ─────────────────────────────────────────────────────────────────────────────
from google.cloud import bigquery
import time
import pandas as pd
from datetime import datetime

# client = bigquery.Client()
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Schema and table name definitions (update OBDATE for each model run)
schema1 = 'risk_mart'
schema2 = 'worktable_data_analysis'
schema3 = 'dap_ds_poweruser_playground'

tab = 'bscore'
ver = 'v25'

OBDATE = '2026-08-17'   # Observation date — update for each scoring run
SDATE  = '2023-01-01'   # Start date for training window
EDATE  = OBDATE

formatted_date = datetime.strptime(SDATE, '%Y-%m-%d').strftime('%Y%m%d')
ob_formatted   = datetime.strptime(OBDATE, '%Y-%m-%d').strftime('%Y%m%d')

al       = f'applied_loans_20230101_{ob_formatted}'
altrans  = f'applied_loans_20210701_{ob_formatted}_trans'
nal      = f'tsa_onboarded_but_never_applied_loan_20230101_{ob_formatted}'

# Intermediate staging tables
bscorebasedata = f'{tab}_base_data_{ver}_{datetime.strptime(SDATE, '%Y-%m-%d').strftime('%Y%m%d')}_{datetime.strptime(OBDATE, '%Y-%m-%d').strftime('%Y%m%d')}'
print(bscorebasedata)
bscoresnapshotcustomerdata        = f'{tab}_applied_loans_customer_information_data_{ver}'
bscoresnapshotcustomertransactiondata = f'{tab}_applied_loans_customer_transaction_data_{ver}'
bscoresnapshotcustomerevent       = f'{tab}_applied_loans_customer_event_data_{ver}'
bscoresnapshotcontactability      = f'{tab}_applied_loans_contactability_{ver}'
bscoresnapshotcombineddata        = f'{tab}_model_applied_loans_{ver}_{ob_formatted}'


step61 = f'{schema2}.{tab}_vb_results_{ver}_{ob_formatted}'
step62 = f'{schema2}.{tab}_chatbot_{ver}_{ob_formatted}'
step63 = f'{schema2}.{tab}_chatbot_fb_{ver}_{ob_formatted}'
step64 = f'{schema2}.{tab}_chatbot_users_{ver}_{ob_formatted}'
step65 = f'{schema2}.{tab}_genesys_chat_{ver}_{ob_formatted}'
step66 = f'{schema2}.{tab}_genesys_email_{ver}_{ob_formatted}'
step67 = f'{schema2}.{tab}_inbound_voice_{ver}_{ob_formatted}'
step68 = f'{schema2}.{tab}_airudder_softcollection_calls_{ver}_{ob_formatted}'
step682 = f'{schema2}.{tab}_cs_vb_cnt_calls_total_{ver}_{ob_formatted}'
step69 = f'{schema2}.{tab}_reach_enhanced_{ver}_{ob_formatted}'
step610 = f'{schema2}.{tab}_ars_results_{ver}_{ob_formatted}'
step611 = f'{schema2}.{tab}_inbound_additional_features_{ver}_{ob_formatted}'
step612 = f'{schema2}.{tab}_outbound_voice_features_{ver}_{ob_formatted}'
step6 = f'{schema2}.{tab}_contact_features_extended_{ver}_{ob_formatted}'
step7 = f'{schema2}.{tab}_contact_genesysdata_{ver}_{ob_formatted}'
step8 = f'{schema2}.{tab}_loan_level_swathi_table_{ver}_{ob_formatted}'
step9 = f'{schema2}.{tab}_ptp_details_{ver}_{ob_formatted}'

step101 = f'{schema2}.{tab}_account_product_portfolio_composition_{ver}_{ob_formatted}'
step102 = f'{schema2}.{tab}_activity_recency_dormancy_{ver}_{ob_formatted}'
step103 = f'{schema2}.{tab}_tx_balance_liquidity_{ver}_{ob_formatted}'
step104 = f'{schema2}.{tab}_bureau_external_scoring_{ver}_{ob_formatted}'
step105 = f'{schema2}.{tab}_completion_settlement_metrics_{ver}_{ob_formatted}'
step106 = f'{schema2}.{tab}_conversion_campaign_engagement_events_{ver}_{ob_formatted}'
step107 = f'{schema2}.{tab}_customer_acquisition_retention_{ver}_{ob_formatted}'
step108 = f'{schema2}.{tab}_customer_lifecycle_engagement_{ver}_{ob_formatted}'
step109 = f'{schema2}.{tab}_demographic_verification_enhanced_{ver}_{ob_formatted}'
step1010 = f'{schema2}.{tab}_deposit_savings_products_{ver}_{ob_formatted}'
step1011 = f'{schema2}.{tab}_derived_ratio_metrics_{ver}_{ob_formatted}'
step1012 = f'{schema2}.{tab}_fraud_compliance_indicators_{ver}_{ob_formatted}'
step1013 = f'{schema2}.{tab}_loan_product_specific_metrics_{ver}_{ob_formatted}'
step1014 = f'{schema2}.{tab}_loan_staging_milestones_{ver}_{ob_formatted}'
step10151 = f'{schema2}.{tab}_payment_behaviour_advanced_{ver}_{ob_formatted}'
step10152 = f'{schema2}.{tab}_payment_behaviour_advanced_2_{ver}_{ob_formatted}'
step1016 = f'{schema2}.{tab}_revenue_fee_decomposition_{ver}_{ob_formatted}'
step1017 = f'{schema2}.{tab}_risk_segmentation_categorization_{ver}_{ob_formatted}'
step1018 = f'{schema2}.{tab}_time_window_cohort_features_{ver}_{ob_formatted}'
step1019 = f'{schema2}.{tab}_transaction_volume_type_breakdown_{ver}_{ob_formatted}'
step111 = f'{schema2}.{tab}_collection_call_features_{ver}_{ob_formatted}'
step112 = f'{schema2}.{tab}_payhinga_features_{ver}_{ob_formatted}'

step12c = f'{schema2}.PH_City_Province_Event_Mapping_{ver}_{ob_formatted}'

step12d = f'{schema2}.{tab}_ph_shock_feature_{ver}_{ob_formatted}'
step14 = f'{schema2}.{tab}_prevloan_less_6dpd_details_{ver}_{ob_formatted}'

step17 = f'{schema2}.{tab}_customer_level_5dpd_details_{ver}_{ob_formatted}'

step20 = f'dap_ds_poweruser_playground.{tab}_yuvashri_features_{ver}_{ob_formatted}'

step22 = f'{schema2}.{tab}_finscore_{ver}_{ob_formatted}'
step23 = f'{schema2}.{tab}_trustingsocialscore_{ver}_{ob_formatted}'
step24 = f'{schema2}.{tab}_bureauinc_score_{ver}_{ob_formatted}'

for step_name, step_value in {
    "step61": step61,
    "step62": step62,
    "step63": step63,
    "step64": step64,
    "step65": step65,
    "step66": step66,
    "step67": step67,
    "step68": step68,
    "step682": step682,
    "step69": step69,
    "step610": step610,
    "step611": step611,
    "step612": step612,
    "step6": step6,
    "step7": step7,
    "step8": step8,
    "step9": step9,
    "step101": step101,
    "step102": step102,
    "step103": step103,
    "step104": step104,
    "step105": step105,
    "step106": step106,
    "step107": step107,
    "step108": step108,
    "step109": step109,
    "step1010": step1010,
    "step1011": step1011,
    "step1012": step1012,
    "step1013": step1013,
    "step1014": step1014,
    "step10151": step10151,
    "step10152": step10152,
    "step1016": step1016,
    "step1017": step1017,
    "step1018": step1018,
    "step1019": step1019,
    "step111": step111,
    "step112": step112,
    "step12c": step12c,
    "step12d": step12d,
    "step14": step14,
    "step17": step17,
    "step20": step20,
    "step22": step22,
    "step23": step23,
    "step24": step24,
}.items():
    print(f"{step_name}: {step_value}")



# Final output tables (split by loan type)
trench3_b_score_model_applied_loans = f'{tab}_model_applied_loans_app_package_{ver}_{ob_formatted}'

print(f"OBDATE: {OBDATE}")
print(f"Output table: {schema2}.{bscoresnapshotcombineddata}")

print(f"Applied Table: {schema1}.{al}")
print(f"Applied Trans Table: {schema1}.{altrans}")
print(f"tsa_onboarded_but_never_applied_loan Table: {schema1}.{nal}")

APP_PACKAGE = f'{tab}_applied_all_dis_loan_app_package_{ver}_{ob_formatted}'
print(f"App Package Table: {schema2}.{APP_PACKAGE}")

App_Feature_Table = f'{tab}_model_app_features_{ver}_{ob_formatted}'
print(f"App Feature Table: {schema2}.{App_Feature_Table}")

cic_feature_table = f'{tab}_cic_features_{ver}_{ob_formatted}'
print(f"CIC Feature Table: {schema2}.{tab}_cic_features_{ver}_{ob_formatted}")

android_devise_score_features = f"{tab}_android_device_score_feature_list_{ver}_{ob_formatted}"
print(f"Android Device Score Feature Table: {schema2}.{android_devise_score_features}")

ios_device_score_features = f"{tab}_ios_device_score_features_{ver}_{ob_formatted}"
print(f"iOS Device Score Feature Table: {schema2}.{ios_device_score_features}")

nitin_event_features = f"{schema2}.{tab}_events_feats_{ver}_{ob_formatted}"
print(f"nitin_event_features Table: {nitin_event_features}")

demo_features = f"{schema3}.{tab}_demo_features_{ver}"
print(f"Demo Feature Table: {demo_features}")
stepfinaltab = f"{schema2}.{tab}_model_full_features_v7_{ver}_{ob_formatted}"
print(f"Final Feature Table: {stepfinaltab}")

finaltab = f"{schema2}.{tab}_model_full_features_model_dataset_{ver}_{ob_formatted}"
print(f"Final Table: {finaltab}")

bscore_base_data_v25_20230101_20260817
step61: worktable_data_analysis.bscore_vb_results_v25_20260817
step62: worktable_data_analysis.bscore_chatbot_v25_20260817
step63: worktable_data_analysis.bscore_chatbot_fb_v25_20260817
step64: worktable_data_analysis.bscore_chatbot_users_v25_20260817
step65: worktable_data_analysis.bscore_genesys_chat_v25_20260817
step66: worktable_data_analysis.bscore_genesys_email_v25_20260817
step67: worktable_data_analysis.bscore_inbound_voice_v25_20260817
step68: worktable_data_analysis.bscore_airudder_softcollection_calls_v25_20260817
step682: worktable_data_analysis.bscore_cs_vb_cnt_calls_total_v25_20260817
step69: worktable_data_analysis.bscore_reach_enhanced_v25_20260817
step610: worktable_data_analysis.bscore_ars_results_v25_20260817
step611: worktable_data_analysis.bscore_inbound_additional_features_v25_20260817
step612: worktable_data_analysis.bscore_outbound_voice_features_v25_20260817
step6: worktable_data_analysis.bscore_contact_features_extended_v

In [3]:
# ── Events source table (created in Step 3 below) ────────────────────────────
# SOURCE_TABLE = f'{schema1}.{tab}_moenagage_events_all_20230101_{ob_formatted}'
SOURCE_TABLE = 'risk_mart.bscore_moenagage_events_all_20230101_training_20260817'

# ── Derived mapping table names ───────────────────────────────────────────────
obs_date_dt    = datetime.strptime(OBDATE, '%Y-%m-%d')
incr_start_dt  = obs_date_dt - timedelta(days=5)          # 6-day incremental window
incr_start_str = incr_start_dt.strftime('%d%m%Y')         # DDMMYYYY (matches existing naming)
incr_end_str   = obs_date_dt.strftime('%d%m%Y')
mapping_incr_table    = f'risk_mart.moengange_customerid_userid_mapping_{incr_start_str}_{incr_end_str}_test'
mapping_unified_table = f'risk_mart.moengange_customerid_userid_mapping_01042023_{incr_end_str}_test'

# ── Summary printout ─────────────────────────────────────────────────────────
print(f'OBDATE                : {OBDATE}')
print(f'bscorebasedata        : {bscorebasedata}')
print(f'SOURCE_TABLE          : {SOURCE_TABLE}')
print(f'nitin_event_features  : {nitin_event_features}')
print(f'mapping_incr_table    : {mapping_incr_table}')
print(f'mapping_unified_table : {mapping_unified_table}')
print(f'finaltab              : {finaltab}')

OBDATE                : 2026-08-17
bscorebasedata        : bscore_base_data_v25_20230101_20260817
SOURCE_TABLE          : risk_mart.bscore_moenagage_events_all_20230101_training_20260817
nitin_event_features  : worktable_data_analysis.bscore_events_feats_v25_20260817
mapping_incr_table    : risk_mart.moengange_customerid_userid_mapping_12082026_17082026_test
mapping_unified_table : risk_mart.moengange_customerid_userid_mapping_01042023_17082026_test
finaltab              : worktable_data_analysis.bscore_model_full_features_model_dataset_v25_20260817


## 3  BigQuery Client

In [5]:
PROJECT_ID = 'prj-prod-dataplatform'
client = bigquery.Client(project=PROJECT_ID)
print(f'BigQuery client ready  →  project: {PROJECT_ID}')

BigQuery client ready  →  project: prj-prod-dataplatform


## 4  Build BigQuery Tables
### 4a  Incremental Mapping Table
Creates a new `customer_id → moengage_user_id` mapping for the observation window.

In [ ]:
sq1 = f"""
CREATE OR REPLACE TABLE {mapping_incr_table} AS
SELECT DISTINCT customer_id, moengage_user_id
FROM `prj-prod-dataplatform.moengage_raw.events_hourly`
WHERE DATE(event_time) BETWEEN '{incr_start_dt.strftime('%Y-%m-%d')}' AND '{OBDATE}';
"""

print(f'Creating: {mapping_incr_table} ...')
job = client.query(sq1)
job.result()

tbl = client.get_table(mapping_incr_table)
print(f'Done  |  columns: {len(tbl.schema)}  |  rows: {tbl.num_rows:,}')

### 4b  Unified Mapping Table (UNION ALL)
Combines all incremental mapping tables into one deduplicated master table.

In [ ]:
sq2 = f"""
CREATE OR REPLACE TABLE {mapping_unified_table} AS
SELECT DISTINCT * FROM (
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_01102024_31122024
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_01012025_08042026
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_09042026_26042026
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_01092024_30092024
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_15032026_10062026
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_11062026_25062026
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_27062026_10072026
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_01042023_30092024
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_26062026_02082026
    UNION ALL
    SELECT * FROM risk_mart.moengange_customerid_userid_mapping_03082026_05082026
    UNION ALL
    SELECT * FROM {mapping_incr_table}
);
"""

print(f'Creating: {mapping_unified_table} ...')
job = client.query(sq2)
job.result()

tbl = client.get_table(mapping_unified_table)
print(f'Done  |  columns: {len(tbl.schema)}  |  rows: {tbl.num_rows:,}')

### 4c  Events Source Table
Joins the base data customer list with MoEngage raw events, applies a 7-day per-observation window,
and deduplicates. This is the main input for feature engineering.

In [ ]:
sq3 = f"""
CREATE OR REPLACE TABLE `{SOURCE_TABLE}`
CLUSTER BY customer_id, digitalLoanAccountId AS

WITH customer_list AS (
    SELECT
        customerId,
        digitalLoanAccountId,
        Observation_date
    FROM {schema2}.{bscorebasedata}
),

event_data AS (
    SELECT
        events.customer_id,
        events.moengage_user_id,
        events.event_time,
        events.event_name,
        cl.digitalLoanAccountId,
        cl.Observation_date
    FROM `prj-prod-dataplatform.moengage_raw.events_hourly` events
    INNER JOIN customer_list cl
        ON events.customer_id = cl.customerId
    WHERE
        events.event_time >= TIMESTAMP(
            DATE_SUB(DATE(cl.Observation_date), INTERVAL 6 DAY),
            'Asia/Manila'
        )
        AND events.event_time < TIMESTAMP(
            DATE_ADD(DATE(cl.Observation_date), INTERVAL 1 DAY),
            'Asia/Manila'
        )
),

final_data AS (
    SELECT *
    FROM event_data
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY
            customer_id,
            moengage_user_id,
            event_time,
            event_name,
            Observation_date
        ORDER BY digitalLoanAccountId
    ) = 1
)

SELECT * FROM final_data;
"""

print(f'Creating: {SOURCE_TABLE} ...')
print('(This query scans events_hourly — may take several minutes)')
t0 = time.time()
job = client.query(sq3)
job.result()
elapsed = time.time() - t0

tbl = client.get_table(SOURCE_TABLE)
print(f'Done in {elapsed:.0f}s  |  columns: {len(tbl.schema)}  |  rows: {tbl.num_rows:,}')

## 5  Load Events Table into Pandas

In [7]:
import sys
sys.path.append(r"D:\Officedata\Feature_Universe_Training")
from Utils.bq_fast_download_updated import bq_to_df
query = f"""
SELECT
    CAST(customer_id AS STRING) AS customer_id,
    moengage_user_id,
    event_time,
    event_name,
    digitalLoanAccountId,
    Observation_date
FROM `{SOURCE_TABLE}`
"""

print(f'Loading {SOURCE_TABLE} ...')
t0 = time.time()
# raw_df = client.query(query).to_dataframe(progress_bar_type = 'tqdm')
raw_df = bq_to_df(query)
elapsed = time.time() - t0

print(f'Loaded {len(raw_df):,} rows in {elapsed:.1f}s')
print(f'Shape : {raw_df.shape}')
print(f'Memory: {raw_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
raw_df.dtypes

Loading risk_mart.bscore_moenagage_events_all_20230101_training_20260817 ...


INFO:Utils.bq_fast_download_updated:Running query (anonymous destination, no named table)
INFO:Utils.bq_fast_download_updated:Query finished (14943779 rows). Anonymous result table: prj-prod-dataplatform._72abd532fd87b7ae4e292515574e1092acf0bf59.anonbbc32b8f411463a40fe8eef2934f5c693f6b2933beecdce0fb15eea65fa33a45
INFO:Utils.bq_fast_download_updated:Extracting results to gs://prod-asia-southeast1-tonik-aiml-workspace/DC/Parquet_files/1b7127fb5af3/tmp_result-*.parquet
INFO:Utils.bq_fast_download_updated:Downloading 50 parquet shard(s) with 16 workers
INFO:Utils.bq_fast_download_updated:Combining 50 local parquet file(s) via DuckDB
INFO:Utils.bq_fast_download_updated:Reading parquet shards into DataFrame (enforce_schema=True)
INFO:Utils.bq_fast_download_updated:DuckDB: memory_limit=12GB, threads=16, spill_dir=/tmp\bq_download_1b7127fb5af3
INFO:Utils.bq_fast_download_updated:Loaded DataFrame with shape (14943779, 6)
INFO:Utils.bq_fast_download_updated:Deleted 50 GCS parquet file(s)


Loaded 14,943,779 rows in 223.9s
Shape : (14943779, 6)
Memory: 2009.0 MB


customer_id                        str
moengage_user_id                   str
event_time              datetime64[us]
event_name                         str
digitalLoanAccountId               str
Observation_date        datetime64[us]
dtype: object

## 6  Preprocess Timestamps

In [8]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. event_time (UTC) -> event_time_manila (Asia/Manila, tz-naive)
    2. Observation_date (Manila date) -> observation_datetime = date + 23:59:00 (tz-naive)
    """
    df = df.copy()

    # event_time: UTC -> Manila, strip tz
    df['event_time'] = pd.to_datetime(df['event_time'], utc=True, errors='coerce')
    df['event_time_manila'] = (
        df['event_time']
        .dt.tz_convert('Asia/Manila')
        .dt.tz_localize(None)
    )
    df.drop(columns=['event_time'], inplace=True)

    # Observation_date -> observation_datetime (Manila date + 23:59:00, tz-naive)
    df['Observation_date'] = pd.to_datetime(df['Observation_date'], errors='coerce')
    df['observation_datetime'] = (
        df['Observation_date'] + pd.Timedelta(hours=23, minutes=59, seconds=0)
    )
    df.drop(columns=['Observation_date'], inplace=True)

    df['customer_id'] = df['customer_id'].astype(str)
    return df


events_df = preprocess(raw_df)
del raw_df   # free raw copy

print(f'Preprocessed shape : {events_df.shape}')
print(f'event_time_manila  : {events_df["event_time_manila"].dtype}')
print(f'observation_datetime: {events_df["observation_datetime"].dtype}')
events_df[['event_time_manila', 'observation_datetime']].head(3)

Preprocessed shape : (14943779, 6)
event_time_manila  : datetime64[us]
observation_datetime: datetime64[us]


,event_time_manila,observation_datetime
0,2026-05-12 16:24:52,2026-05-12 23:59:00
1,2026-05-14 10:54:40,2026-05-15 23:59:00
2,2026-05-13 15:31:38,2026-05-16 23:59:00


## 7  Feature Engineering
### 7a  Import Feature Package

In [9]:
EVENTS_FEATURES_DIR = r'D:\Officedata\Feature_Universe_Training\Events_Features'
if EVENTS_FEATURES_DIR not in sys.path:
    sys.path.insert(0, EVENTS_FEATURES_DIR)

from B_Score_Events_FE_codebase_v20260818 import create_all_features

print('Feature package imported successfully.')
print()
print('Modules (11 compute functions):')
print('  features_base     — session, entropy, app-open, login counts')
print('  features_loan_app — QL funnel depth, sales screens, offer, disbursement')
print('  features_flexup   — FlexUp funnel, offers, timing')
print('  features_silrel   — SIL-REL funnel, purchase success')
print('  features_rl       — Re-Lending funnel, offer, disbursement')
print('  features_fl       — FL / FLXL / UIL screens')
print('  features_payment  — payment attempts, GCash, reminders, recency')
print('  features_comms    — email, SMS, push, in-app engagement')
print('  features_financial — topup, cards, GCash linking, stash, send')
print('  features_support  — help, chatbot, FAQ, hotline')
print('  features_session  — session count/duration, ToD, weekend ratio')

Feature package imported successfully.

Modules (11 compute functions):
  features_base     — session, entropy, app-open, login counts
  features_loan_app — QL funnel depth, sales screens, offer, disbursement
  features_flexup   — FlexUp funnel, offers, timing
  features_silrel   — SIL-REL funnel, purchase success
  features_rl       — Re-Lending funnel, offer, disbursement
  features_fl       — FL / FLXL / UIL screens
  features_payment  — payment attempts, GCash, reminders, recency
  features_comms    — email, SMS, push, in-app engagement
  features_financial — topup, cards, GCash linking, stash, send
  features_support  — help, chatbot, FAQ, hotline
  features_session  — session count/duration, ToD, weekend ratio


### 7b  Build Loan Metadata & Pre-Group Events

In [ ]:
def build_loans_meta(df: pd.DataFrame) -> pd.DataFrame:
    """One row per digitalLoanAccountId with its stable identifiers."""
    return (
        df.groupby('digitalLoanAccountId', sort=False)
        .agg(
            customer_id          = ('customer_id',          'first'),
            moengage_user_id     = ('moengage_user_id',     'first'),
            observation_datetime = ('observation_datetime', 'first'),
        )
        .reset_index()
    )


loans_meta = build_loans_meta(events_df)

# Pre-group by loan for O(1) per-loan event lookup (key performance gain vs sequential scan)
grouped = events_df.groupby(['customer_id', 'observation_datetime'], sort=False)

print(f'Total loans         : {len(loans_meta):,}')
print(f'Unique customers    : {events_df["customer_id"].nunique():,}')
print(f'Unique loan accounts: {events_df["digitalLoanAccountId"].nunique():,}')
loans_meta.head(3)

### 7c  Parallel Feature Generation
Uses `ThreadPoolExecutor` for concurrent loan processing.
The pre-grouped DataFrame ensures O(1) event lookup per loan.

In [ ]:
# ── Worker function ────────────────────────────────────────────────────────────
def process_single_loan(loan_id, obs_dt, cust_id, moe_uid, loan_events_df):
    """Apply 7-day window, compute all features, return result dict."""
    row_base = {
        'customer_id'          : cust_id,
        'digitalLoanAccountId' : loan_id,
        'moengage_user_id'     : moe_uid,
        'observation_datetime' : obs_dt,
    }
    try:
        window_start = obs_dt - pd.Timedelta(hours=168)
        window_df = loan_events_df[
            (loan_events_df['event_time_manila'] >= window_start) &
            (loan_events_df['event_time_manila'] <= obs_dt)
        ].copy()
        window_df.sort_values('event_time_manila', inplace=True)
        window_df['observation_datetime'] = obs_dt

        feats = create_all_features(window_df) if len(window_df) > 0 else {}
        row_base.update(feats)
    except Exception as exc:
        print(f'  ERROR  loan={loan_id}  cust={cust_id}: {exc}')
    return row_base


# ── Parallel execution ─────────────────────────────────────────────────────────
N_WORKERS = min(8, (os.cpu_count() or 4))
print(f'Running with {N_WORKERS} worker threads across {len(loans_meta):,} loans ...')

all_results = []
t0 = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {}
    for row in loans_meta.itertuples(index=False):
        loan_id = row.digitalLoanAccountId
        try:
            loan_events = grouped.get_group(loan_id)
        except KeyError:
            loan_events = pd.DataFrame(columns=events_df.columns)
        futures[executor.submit(
            process_single_loan,
            loan_id, row.observation_datetime,
            row.customer_id, row.moengage_user_id,
            loan_events
        )] = loan_id

    for future in tqdm(as_completed(futures), total=len(futures), desc='Processing loans'):
        result = future.result()
        if result:
            all_results.append(result)

elapsed = time.time() - t0
print(f'Processed {len(all_results):,} loans in {elapsed:.1f}s '
      f'({len(all_results) / elapsed:.1f} loans/sec)')

### 7d  Assemble Results DataFrame

In [ ]:
results_df = pd.DataFrame(all_results)

# Ensure identifier columns are first
id_cols  = ['customer_id', 'digitalLoanAccountId', 'moengage_user_id', 'observation_datetime']
feat_cols = [c for c in results_df.columns if c not in id_cols]
results_df = results_df[id_cols + feat_cols]

print(f'Shape (loan-level)  : {results_df.shape}')
print(f'Feature columns     : {len(feat_cols)}')
results_df[id_cols].head(3)

### 7e  Deduplicate to 1 Row per `customer_id × observation_date`
A customer may have multiple `digitalLoanAccountId` values sharing the same observation date.
We collapse these to a single row per customer-date, taking the first non-null value
for each feature (sorted by `digitalLoanAccountId` for determinism).

In [ ]:
# Convert observation_datetime -> observation_date (date only)
results_df['observation_date'] = pd.to_datetime(results_df['observation_datetime']).dt.date

# Sort for deterministic first() pick
results_df.sort_values(
    ['customer_id', 'observation_date', 'digitalLoanAccountId'],
    inplace=True
)

# Collapse to 1 row per customer × observation_date
customer_date_df = (
    results_df
    .groupby(['customer_id', 'observation_date'], as_index=False)
    .first()
)

# Drop loan-level columns that are no longer meaningful at customer-date level
customer_date_df.drop(columns=['digitalLoanAccountId', 'observation_datetime'], inplace=True)

# Final column order: customer_id | observation_date | moengage_user_id | features
final_id_cols  = ['customer_id', 'observation_date', 'moengage_user_id']
final_feat_cols = [c for c in customer_date_df.columns if c not in final_id_cols]
customer_date_df = customer_date_df[final_id_cols + final_feat_cols]

# Sanity check: must be exactly 1 row per customer-date
dupe_count = customer_date_df.groupby(['customer_id', 'observation_date']).size().max()
assert dupe_count == 1, f'Duplicate customer-date rows found! max group size = {dupe_count}'

print(f'Shape (customer-date): {customer_date_df.shape}')
print(f'Feature columns      : {len(final_feat_cols)}')
print(f'Null rate (%)        : {customer_date_df.isnull().mean().mean() * 100:.2f}')
customer_date_df[final_id_cols].head(5)

## 8  Save to BigQuery

In [ ]:
# ── Write to BigQuery ──────────────────────────────────────────────────────────
dest_dataset, dest_table_name = nitin_event_features.split('.', 1)
dest_table_ref = f'{dest_dataset}.{dest_table_name}'

job_config = bigquery.LoadJobConfig(
    write_disposition='WRITE_TRUNCATE',
    autodetect=True,
)

print(f'Writing to BigQuery: {nitin_event_features} ...')
t0 = time.time()
job = client.load_table_from_dataframe(
    customer_date_df,
    nitin_event_features,
    job_config=job_config,
)
job.result()
elapsed = time.time() - t0

bq_tbl = client.get_table(nitin_event_features)
print(f'BQ write done in {elapsed:.1f}s  |  rows: {bq_tbl.num_rows:,}  |  cols: {len(bq_tbl.schema)}')

# ── Local CSV backup ───────────────────────────────────────────────────────────
csv_path = os.path.join(EVENTS_FEATURES_DIR, f'bscore_event_features_{ob_formatted}.csv')
customer_date_df.to_csv(csv_path, index=False)
print(f'CSV backup          : {csv_path}')

## 9  Validation / QC

In [ ]:
print('=== QC Summary ===')
print(f'Rows (customer-date pairs)  : {len(customer_date_df):,}')
print(f'Unique customers            : {customer_date_df["customer_id"].nunique():,}')
print(f'Unique observation dates    : {customer_date_df["observation_date"].nunique()}')
print(f'Feature columns             : {len(final_feat_cols)}')
print()

# Null rate per column (top 10 worst)
null_rate = customer_date_df.isnull().mean().sort_values(ascending=False).head(10)
print('Top 10 columns by null rate:')
print(null_rate.to_string())
print()

# Spot-check key features
spot_cols = [
    'customer_id', 'observation_date',
    'total_events_cnt_7d', 'ql_funnel_max_depth_7d',
    'repayment_success_cnt_7d', 'session_cnt_7d',
]
available = [c for c in spot_cols if c in customer_date_df.columns]
print('Sample rows (spot-check):')
customer_date_df[available].sample(min(5, len(customer_date_df)), random_state=42)